# Complete BaBar isobar CP closure with Square Dalitz normalization

This notebook implements the complete nominal BaBar signal model of Phys. Rev. D 78, 012004 (2008), arXiv:0803.4451. The Cartesian CP convention is
\[c_j^+=(x_j+\Delta x_j)+i(y_j+\Delta y_j),\qquad c_j^-=(x_j-\Delta x_j)+i(y_j-\Delta y_j).\]

The fit is performed in the joint space of charge and Dalitz coordinates,
\[p(\Phi,q)=\frac{|A_q(\Phi)|^2}{I_+ + I_-},\qquad I_\pm=\int |A_\pm(\Phi)|^2d\Phi.\]

For this B-decay benchmark the normalization is evaluated on a **Square Dalitz Plot (SDP)** grid. Particle ordering is `(1,2,3)=(K^\pm,\pi^\pm,\pi^\mp)` and the SDP pair is `(1,3)`, so the transformed mass is $m_{13}=m(K^\pm\pi^\mp)$. The ordinary Dalitz plane remains
\[s_{13}=m^2(K^\pm\pi^\mp),\qquad s_{23}=m^2(\pi^+\pi^-).\]

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, Resonance, SquareDalitzGrid, enable_x64,
    invariants_to_square_dalitz, weighted_resample,
)
from dalitzplotfitter.likelihood import CPJointNLL

enable_x64()

## 1. BaBar Table-I Cartesian CP coefficients

In [ ]:
TABLE_I = {
    # name: (x, y, dx, dy, fixed_xy, fixed_cp)
    'Kstar892':    ( 1.000,  0.000, -0.017, -0.238, True,  False),
    'KpiS':        ( 1.718, -0.727, -0.154, -0.285, False, False),
    'rho770':      ( 0.683, -0.025, -0.160,  0.169, False, False),
    'f0_980':      (-0.220,  1.203, -0.109,  0.047, False, False),
    'chic0':       (-0.263,  0.180, -0.033, -0.007, False, False),
    'NR':          (-0.594,  0.068,  0.000,  0.000, False, True),
    'K2star1430':  (-0.301,  0.424,  0.032,  0.007, False, False),
    'omega782':    (-0.058,  0.100,  0.000,  0.000, False, True),
    'f2_1270':     (-0.193,  0.110, -0.089,  0.125, False, False),
    'fX1300':      (-0.290, -0.136,  0.024,  0.056, False, False),
}

def cp_coefficient(name):
    x, y, dx, dy, fixed_xy, fixed_cp = TABLE_I[name]
    def p(suffix, value, fixed=False, bound=3.0, step=0.02):
        return Parameter.coefficient(
            f'{name}.{suffix}', value, owner=name, fixed=fixed,
            bounds=(-bound, bound), step=step,
        )
    return CPRealImag(
        p('x', x, fixed_xy), p('y', y, fixed_xy),
        p('dx', dx, fixed_cp, 1.5, 0.01), p('dy', dy, fixed_cp, 1.5, 0.01),
    )

coeff = {name: cp_coefficient(name) for name in TABLE_I}

## 2. Complete nominal dynamical model

The ten coherent terms are $K^*(892)^0$, LASS $K\pi$ S-wave, $K_2^*(1430)^0$, $\rho(770)^0$, $\omega(782)$, $f_0(980)$, $f_2(1270)$, $f_X(1300)$, $\chi_{c0}$ and a constant nonresonant amplitude.

In [ ]:
channel_plus  = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
channel_minus = DecayChannel('B-', ('K-', 'pi-', 'pi+'))

def build_model(channel, charge):
    c = lambda name: coeff[name].for_charge(charge)
    R = 4.0
    return DecayModel(channel, [
        Resonance('Kstar892',   (0,2), c('Kstar892'),   mass=0.8958, width=0.0474, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('KpiS',       (0,2), c('KpiS'),       lineshape=LASS(2.07,3.32,1.8), mass=1.425, width=0.270, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('rho770',     (1,2), c('rho770'),     mass=0.7753, width=0.1491, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f0_980',     (1,2), c('f0_980'),     lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('chic0',      (1,2), c('chic0'),      mass=3.4147, width=0.0105, spin=0, resonance_radius=R, parent_radius=R),
        NonResonant(c('NR'), name='NR'),
        Resonance('K2star1430', (0,2), c('K2star1430'), mass=1.4324, width=0.109, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('omega782',   (1,2), c('omega782'),   mass=0.78265, width=0.00849, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f2_1270',    (1,2), c('f2_1270'),    mass=1.2755, width=0.1867, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('fX1300',     (1,2), c('fX1300'),     mass=1.479, width=0.080, spin=0, resonance_radius=R, parent_radius=R),
    ], normalization_resolution=350)

model_plus = build_model(channel_plus, +1)
model_minus = build_model(channel_minus, -1)
truth = {p.name: float(p.value) for p in model_plus.parameters}
print('free parameters:', sum(not p.fixed for p in model_plus.parameters))

## 3. Square-Dalitz normalization grid

The regular SDP midpoint grid is uniform in $(m',\theta')\in[0,1]^2$. The integration weights are the event-by-event Jacobian $|\partial(s_{13},s_{12})/\partial(m',\theta')|$. The same physical integral is therefore used by the amplitude cache and by the joint CP normalization.

In [ ]:
SDP_RESOLUTION = 450
SDP_PAIR = (0, 2)  # (1,3) in one-based notation: K± pi∓

square_norm_plus = SquareDalitzGrid(
    channel_plus.parent_mass, channel_plus.daughter_masses,
    resolution=SDP_RESOLUTION, pair=SDP_PAIR,
).sample()
square_norm_minus = SquareDalitzGrid(
    channel_minus.parent_mass, channel_minus.daughter_masses,
    resolution=SDP_RESOLUTION, pair=SDP_PAIR,
).sample()

mp_grid, tp_grid = invariants_to_square_dalitz(
    square_norm_plus.s12, square_norm_plus.s13, square_norm_plus.s23,
    mother_mass=channel_plus.parent_mass, masses=channel_plus.daughter_masses, pair=SDP_PAIR,
)
print('normalization points:', square_norm_plus.size)
print('mprime range:', float(mp_grid.min()), float(mp_grid.max()))
print('thetaprime range:', float(tp_grid.min()), float(tp_grid.max()))

## 4. Generate one joint charge + Dalitz toy

The proposal events still come from `PhaseSpaceMC`, but both the component normalization and the charge fractions are evaluated with the Square Dalitz grid. The generated charge split follows
\[P(+)=I_+/(I_++I_-),\qquad P(-)=I_-/(I_++I_-).\]

In [ ]:
N_POOL, N_TOTAL = 500_000, 120_000
pool_plus = model_plus.generate_phase_space(N_POOL, seed=78012004)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=78012005)

proposal_cache_plus = model_plus.prepare_cache(pool_plus, normalization_sample=square_norm_plus)
proposal_cache_minus = model_minus.prepare_cache(pool_minus, normalization_sample=square_norm_minus)

w_plus = pool_plus.weights * proposal_cache_plus.intensity(truth)
w_minus = pool_minus.weights * proposal_cache_minus.intensity(truth)
I_plus = float(proposal_cache_plus.normalization(truth))
I_minus = float(proposal_cache_minus.normalization(truth))
p_plus = I_plus / (I_plus + I_minus)

rng = np.random.default_rng(78012006)
N_PLUS = rng.binomial(N_TOTAL, p_plus)
N_MINUS = N_TOTAL - N_PLUS
print(f'I+={I_plus:.6f}, I-={I_minus:.6f}, P(+)={p_plus:.5f}')
print(f'N+={N_PLUS}, N-={N_MINUS}, raw asym={(N_MINUS-N_PLUS)/N_TOTAL:+.5f}')

toy_plus = weighted_resample(jax.random.key(78012007), pool_plus, w_plus, N_PLUS, replace=True)
toy_minus = weighted_resample(jax.random.key(78012008), pool_minus, w_minus, N_MINUS, replace=True)

## 5. Ordinary Dalitz plots: $s_{13}$ versus $s_{23}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
for ax, toy, title in [(axes[0], toy_plus, r'$B^+$'), (axes[1], toy_minus, r'$B^-$')]:
    h = ax.hist2d(np.asarray(toy.s13), np.asarray(toy.s23), bins=110)
    fig.colorbar(h[3], ax=ax, label='events')
    ax.set(
        xlabel=r'$s_{13}=m^2(K^\pm\pi^\mp)$ [GeV$^2$]',
        ylabel=r'$s_{23}=m^2(\pi^+\pi^-)$ [GeV$^2$]',
        title=title,
    )
plt.show()

## 6. Square Dalitz plots

The same events are shown in $(m',\theta')$. Narrow structures near ordinary Dalitz boundaries are spread over a larger visible area.

In [ ]:
def square_coordinates(sample, channel):
    return invariants_to_square_dalitz(
        sample.s12, sample.s13, sample.s23,
        mother_mass=channel.parent_mass, masses=channel.daughter_masses, pair=SDP_PAIR,
    )

mp_plus, tp_plus = square_coordinates(toy_plus, channel_plus)
mp_minus, tp_minus = square_coordinates(toy_minus, channel_minus)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, mp, tp, title in [
    (axes[0], mp_plus, tp_plus, r'$B^+$ Square Dalitz'),
    (axes[1], mp_minus, tp_minus, r'$B^-$ Square Dalitz'),
]:
    h = ax.hist2d(np.asarray(mp), np.asarray(tp), bins=100, range=((0,1),(0,1)))
    fig.colorbar(h[3], ax=ax, label='events')
    ax.set(xlabel=r'$m^\prime$', ylabel=r'$\theta^\prime$', title=title, xlim=(0,1), ylim=(0,1))
plt.show()

## 7. Local CP-asymmetry maps in both coordinate systems

In [ ]:
def asymmetry_map(xp, yp, xm, ym, bins, ranges):
    Hp, xe, ye = np.histogram2d(np.asarray(xp), np.asarray(yp), bins=bins, range=ranges)
    Hm, _, _ = np.histogram2d(np.asarray(xm), np.asarray(ym), bins=(xe, ye))
    return xe, ye, (Hm - Hp) / np.maximum(Hm + Hp, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
xe, ye, asym_dp = asymmetry_map(toy_plus.s13, toy_plus.s23, toy_minus.s13, toy_minus.s23, 65, None)
im = axes[0].pcolormesh(xe, ye, asym_dp.T, vmin=-1, vmax=1)
fig.colorbar(im, ax=axes[0], label=r'$(N_- - N_+)/(N_- + N_+)$')
axes[0].set(xlabel=r'$s_{13}$ [GeV$^2$]', ylabel=r'$s_{23}$ [GeV$^2$]', title='Dalitz CP asymmetry')

xe, ye, asym_sdp = asymmetry_map(mp_plus, tp_plus, mp_minus, tp_minus, 65, ((0,1),(0,1)))
im = axes[1].pcolormesh(xe, ye, asym_sdp.T, vmin=-1, vmax=1)
fig.colorbar(im, ax=axes[1], label=r'$(N_- - N_+)/(N_- + N_+)$')
axes[1].set(xlabel=r'$m^\prime$', ylabel=r'$\theta^\prime$', title='Square-Dalitz CP asymmetry', xlim=(0,1), ylim=(0,1))
plt.show()

## 8. One-dimensional projections in Dalitz and Square-Dalitz coordinates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
projection_data = [
    (toy_plus.s13, toy_minus.s13, r'$s_{13}=m^2(K\pi)$ [GeV$^2$]'),
    (toy_plus.s23, toy_minus.s23, r'$s_{23}=m^2(\pi\pi)$ [GeV$^2$]'),
    (mp_plus, mp_minus, r'$m^\prime$'),
    (tp_plus, tp_minus, r'$\theta^\prime$'),
]
for ax, (vp, vm, label) in zip(axes.ravel(), projection_data):
    vp, vm = np.asarray(vp), np.asarray(vm)
    bins = np.linspace(min(vp.min(), vm.min()), max(vp.max(), vm.max()), 100)
    ax.hist(vp, bins=bins, histtype='step', label=r'$B^+$')
    ax.hist(vm, bins=bins, histtype='step', label=r'$B^-$')
    ax.set(xlabel=label, ylabel='events')
    ax.legend()
plt.show()

## 9. Joint CP likelihood with one global SDP normalization

Both charge caches use the same square-coordinate convention and `CPJointNLL` fits
\[-\log\mathcal L=-\sum_+\log|A_+|^2-\sum_-\log|A_-|^2+(N_++N_-)\log(I_++I_-).\]

In [ ]:
cache_plus = model_plus.prepare_cache(toy_plus, normalization_sample=square_norm_plus)
cache_minus = model_minus.prepare_cache(toy_minus, normalization_sample=square_norm_minus)
objective = CPJointNLL(cache_plus, cache_minus)
parameters = model_plus.parameters
fitter = Minimizer(objective, parameters, tolerance=1e-5, verbose=1)
start = fitter.random_start(seed=20260830)
result = fitter.fit(start_values=start, simplex=False, ncall=50000)
print(result.fmin)
print('charge probabilities truth:', tuple(float(v) for v in objective.charge_probabilities(truth)))

## 10. Pulls of all floating Cartesian parameters

In [ ]:
free = [p for p in parameters if not p.fixed]
fit = {p.name: float(result.values[p.name]) for p in free}
err = {p.name: float(result.errors[p.name]) for p in free}
fit_all = {**truth, **fit}
pulls = np.array([(fit[p.name]-truth[p.name])/err[p.name] for p in free])

print(f"{'parameter':18s} {'truth':>9s} {'start':>9s} {'fit':>9s} {'err':>9s} {'pull':>8s}")
for p, pull in zip(free, pulls):
    print(f"{p.name:18s} {truth[p.name]:9.4f} {start[p.name]:9.4f} {fit[p.name]:9.4f} {err[p.name]:9.4f} {pull:8.2f}")

fig, ax = plt.subplots(figsize=(13,5), constrained_layout=True)
ax.axhline(0, lw=.8); ax.axhline(1, ls='--', lw=.7); ax.axhline(-1, ls='--', lw=.7)
ax.scatter(np.arange(len(free)), pulls)
ax.set_xticks(np.arange(len(free)), [p.name for p in free], rotation=75, ha='right')
ax.set(ylabel='pull', title='Closure pulls: all floating CP parameters')
plt.show()

## 11. Component $A_{CP}$ and Argand closure

In [ ]:
def complex_pair(c, values):
    return complex(c.for_charge(+1).value(values)), complex(c.for_charge(-1).value(values))

def acp(c, values):
    cp, cm = complex_pair(c, values)
    return (abs(cm)**2 - abs(cp)**2) / (abs(cm)**2 + abs(cp)**2)

names = list(coeff)
acp_truth = np.array([acp(coeff[n], truth) for n in names])
acp_fit = np.array([acp(coeff[n], fit_all) for n in names])

fig, ax = plt.subplots(figsize=(11,5), constrained_layout=True)
x = np.arange(len(names)); ax.axhline(0, lw=.8)
ax.scatter(x, acp_truth, marker='x', s=80, label='truth')
ax.scatter(x, acp_fit, marker='o', s=45, label='fit')
ax.set_xticks(x, names, rotation=45, ha='right')
ax.set(ylabel=r'$A_{CP}^j$', title='Component CP asymmetries')
ax.legend(); plt.show()

fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, charge, title in [(axes[0],+1,r'$B^+$ coefficients'),(axes[1],-1,r'$B^-$ coefficients')]:
    for name in names:
        zt = complex(coeff[name].for_charge(charge).value(truth))
        zf = complex(coeff[name].for_charge(charge).value(fit_all))
        ax.plot([zt.real,zf.real],[zt.imag,zf.imag],'-',alpha=.5)
        ax.scatter([zt.real],[zt.imag],marker='x',s=55)
        ax.scatter([zf.real],[zf.imag],marker='o',s=28)
        ax.text(zf.real,zf.imag,name,fontsize=8)
    ax.axhline(0,lw=.6); ax.axvline(0,lw=.6); ax.set(xlabel='Re(c)',ylabel='Im(c)',title=title)
plt.show()

## 12. Truth vs fitted global charge fraction

In [ ]:
p_truth = tuple(float(v) for v in objective.charge_probabilities(truth))
p_fit = tuple(float(v) for v in objective.charge_probabilities(fit_all))
print(f'P(+): truth={p_truth[0]:.6f}, fit={p_fit[0]:.6f}, observed={N_PLUS/N_TOTAL:.6f}')
print(f'P(-): truth={p_truth[1]:.6f}, fit={p_fit[1]:.6f}, observed={N_MINUS/N_TOTAL:.6f}')

fig, ax = plt.subplots(figsize=(7,4), constrained_layout=True)
x=np.arange(2); width=.25
ax.bar(x-width,[p_truth[0],p_truth[1]],width,label='truth')
ax.bar(x,[p_fit[0],p_fit[1]],width,label='fit')
ax.bar(x+width,[N_PLUS/N_TOTAL,N_MINUS/N_TOTAL],width,label='toy')
ax.set_xticks(x,[r'$B^+$',r'$B^-$']); ax.set(ylabel='charge fraction',title='Global CP rate information')
ax.legend(); plt.show()

## 13. Post-fit projection checks

The same phase-space proposal pools are reweighted with the fitted amplitudes. Their predicted charge yields use the fitted joint probabilities, so these curves test both Dalitz shape and global CP normalization.

In [ ]:
fit_w_plus = np.asarray(pool_plus.weights * proposal_cache_plus.intensity(fit_all))
fit_w_minus = np.asarray(pool_minus.weights * proposal_cache_minus.intensity(fit_all))
fit_p_plus, fit_p_minus = p_fit

def weighted_projection(ax, toy_values, pool_values, pool_weights, expected_events, label):
    toy_values = np.asarray(toy_values); pool_values = np.asarray(pool_values)
    bins = np.linspace(min(toy_values.min(), pool_values.min()), max(toy_values.max(), pool_values.max()), 90)
    counts, edges = np.histogram(toy_values, bins=bins)
    prediction, _ = np.histogram(pool_values, bins=bins, weights=pool_weights)
    prediction = prediction * expected_events / prediction.sum()
    centers = 0.5*(edges[:-1]+edges[1:])
    ax.errorbar(centers, counts, yerr=np.sqrt(np.maximum(counts,1)), fmt='.', label='toy')
    ax.step(centers, prediction, where='mid', label='fit')
    ax.set(xlabel=label, ylabel='events'); ax.legend()

fig, axes = plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
weighted_projection(axes[0,0],toy_plus.s13,pool_plus.s13,fit_w_plus,N_TOTAL*fit_p_plus,r'$B^+: s_{13}$')
weighted_projection(axes[0,1],toy_plus.s23,pool_plus.s23,fit_w_plus,N_TOTAL*fit_p_plus,r'$B^+: s_{23}$')
weighted_projection(axes[1,0],toy_minus.s13,pool_minus.s13,fit_w_minus,N_TOTAL*fit_p_minus,r'$B^-: s_{13}$')
weighted_projection(axes[1,1],toy_minus.s23,pool_minus.s23,fit_w_minus,N_TOTAL*fit_p_minus,r'$B^-: s_{23}$')
plt.show()